In [ ]:
import numpy as np

from common.time_layers import *
import matplotlib.pyplot as plt
from common.optimizer import SGD
from dataset import ptb

In [ ]:
class SimpleRNNLM:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        self.vocab_size = vocab_size
        self.wordvec_size = wordvec_size
        self.hidden_size = hidden_size

        rn = np.random.randn

        ebd_W = (rn(vocab_size, wordvec_size) / 100).astype('f')
        rnn_Wx = (rn(wordvec_size, hidden_size) / np.sqrt(wordvec_size)).astype('f')
        rnn_Wh = (rn(hidden_size, hidden_size) / np.sqrt(hidden_size)).astype('f')
        rnn_b = np.zeros(hidden_size).astype('f')

        affine_W = (rn(hidden_size, vocab_size) / np.sqrt(hidden_size)).astype('f')
        affine_b = np.zeros(vocab_size).astype('f')

        self.layers = [
            TimeEmbedding(ebd_W),
            TimeRNN(rnn_Wx, rnn_Wh, rnn_b, stateful=True),
            TimeAffine(affine_W, affine_b)
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.rnn_layer = self.layers[1]

        self.params = []
        self.grads = []

        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads



    def forward(self, xs, ts):
        for layer in self.layers:
            xs = layer.forward(xs)
        loss = self.loss_layer.forward(xs, ts)
        return loss

    def backward(self, dout = 1):
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout



In [ ]:
batch_size = 10
wordvec_size = 100
hidden_size  = 100
time_size = 5
lr = 0.1
max_epoch = 100

corpus, word_to_id, id_to_word = ptb.load_data()
print(type(corpus))
print(f'corpus size: {corpus.shape}')
print(corpus[:500])

In [ ]:
vocab_size = int(max(corpus) + 1)
print(f'vocab_size: {vocab_size}')

In [ ]:
xs = corpus[:-1]
ts = corpus[1:]
print(f'xs size: {xs.shape}, ts size: {ts.shape}')

In [ ]:
data_size = len(xs)
max_iters = data_size // (batch_size * time_size)
print(f'max_iters: {max_iters}')
time_idx = 0
total_loss = 0
loss_count = 0
ppl_list = []

model = SimpleRNNLM(vocab_size, wordvec_size, hidden_size)
optimzer = SGD(lr)


In [ ]:
jump = (corpus.shape[0] - 1) // batch_size
print(f'jump: {jump}')
offsets = [ i * jump for i in range(batch_size)]
print(f'offsets size: {len(offsets)}')
print(offsets)

In [ ]:
for epoch in range(max_epoch):

    for iter in range(max_iters):
        print(f'iter {iter} in epoch {epoch} started')
        batch_x = np.empty((batch_size, time_size), dtype = 'i')
        batch_t = np.empty((batch_size, time_size), dtype = 'i')

        for t in range(time_size):
            for i, offset in enumerate(offsets):
                batch_x[i, t] = xs[(offset + time_idx) % data_size]
                batch_t[i, t] = ts[(offset + time_idx) % data_size]
            time_idx += 1

        loss = model.forward(batch_x, batch_t)
        model.backward()
        optimzer.update(model.params, model.grads)
        total_loss += loss
        loss_count += 1
        print(f'iter {iter} in epoch {epoch} done, {total_loss}, {loss_count}')

    ppl = np.exp(total_loss / loss_count)
    print(f'epoch: {epoch}, pelexity: {ppl}')
    ppl_list.append(float(ppl))
    total_loss = 0
    loss_count = 0

